Base idea from [https://www.kaggle.com/code/barnobarno/test-time-training-tt](https://www.kaggle.com/code/barnobarno/test-time-training-tt)

25/10/13 Update: remove all private resources

Changes:

1. Base model is llama-3.2-3b-instruct
2. Use 5% of hidden test data for pseudo training
3. r=64, lora_alpha=128
4. Use bf16 in deepspeed
5. Add bnb config for non gptq model

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'deepspeed==0.17.4' -q
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

# Pseudo training

### changes, using toy train/val data,
### fraction of data changed to use toy data properly, changed eval steps,inference taken out, order of celles changes, dispatch_batches added


In [1]:
import pandas as pd

# Read original train.csv
train = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/train.csv')

# Split into toy train and toy test
toy_train = train.iloc[:200].copy()  # First 100 rows
toy_test = train.iloc[500:1100].copy()  # Rows 50-150 (50 overlap with train)

# Save
toy_train.to_csv('train.csv', index=False)
toy_test.to_csv('test.csv', index=False)

print(f"Created toy train.csv: {len(toy_train)} rows")
print(f"Created toy test.csv: {len(toy_test)} rows")
print(f"Expected overlap: 50 rows (indices 50-99)")

Created toy train.csv: 200 rows
Created toy test.csv: 600 rows
Expected overlap: 50 rows (indices 50-99)


In [2]:
%%writefile constants.py

seed = 0

base_model_path = "/kaggle/input/jigsaw-pretrain-public/pytorch/llama-3.2-3b-instruct/1"
pretrain_lora_path = None
lora_path = "/kaggle/working/pseudo_lora"
use_gptq = "gptq" in base_model_path

positive = "Yes"
negative = "No"
judge_words = "Violation:"
system_prompt = '''You are given a comment from reddit and a rule.
Your task is to classify whether the comment violates the rule.
Only respond Yes/No.'''

frac = .5
use_train = False

eval_steps = 10
train_csv_path = "train.csv"
test_csv_path = "test.csv"

import kagglehub

deterministic = kagglehub.package_import('wasupandceacar/deterministic').deterministic
deterministic.init_all(seed)

Overwriting constants.py


In [33]:
%%writefile utils.py

import numpy as np
import pandas as pd
from datasets import Dataset
from constants import *

def build_prompt(row):
  return f"""{system_prompt}
Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{judge_words} Yes
2) {row["negative_example"]}
{judge_words} No
Comment: {row["body"]}
{judge_words}"""

def get_eval_df():
  train_df = pd.read_csv(train_csv_path)

  import torch.distributed as dist
  if not dist.is_initialized() or dist.get_rank() == 0:
      print(f"Validation size: {len(train_df)}")

  train_df["positive_example"] = np.where(
      np.random.rand(len(train_df)) < 0.5,
      train_df["positive_example_1"],
      train_df["positive_example_2"]
  )
  train_df["negative_example"] = np.where(
      np.random.rand(len(train_df)) < 0.5,
      train_df["negative_example_1"],
      train_df["negative_example_2"]
  )

  return train_df

def get_df():
  merge = list()
  if use_train:
      train_dataset = pd.read_csv(train_csv_path)
      train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                              "positive_example_1", "positive_example_2",
                              "negative_example_1", "negative_example_2"]].copy()
      train_df["positive_example"] = np.where(np.random.rand(len(train_df)) < 0.5, train_df["positive_example_1"], train_df["positive_example_2"])
      train_df["negative_example"] = np.where(np.random.rand(len(train_df)) < 0.5, train_df["negative_example_1"], train_df["negative_example_2"])
      train_df.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], inplace=True)
      merge.append(train_df)

  test_dataset = pd.read_csv(test_csv_path)
  train_dataset = pd.read_csv(train_csv_path)

  train_bodies = set(train_dataset['body'].values)

  mask = ~(
      test_dataset['positive_example_1'].isin(train_bodies) |
      test_dataset['positive_example_2'].isin(train_bodies) |
      test_dataset['negative_example_1'].isin(train_bodies) |
      test_dataset['negative_example_2'].isin(train_bodies)
  )

  original_len = len(test_dataset)
  test_dataset = test_dataset[mask].copy()

  import torch.distributed as dist
  rank_print = not dist.is_initialized() or dist.get_rank() == 0

  if rank_print:
      print(f"Test: removed {original_len - len(test_dataset)} rows with examples overlapping train bodies")
      print(f"Test after filtering: {len(test_dataset)}")

  test_dataset = test_dataset.groupby('rule', group_keys=False).apply(lambda x: x.sample(frac=frac, random_state=seed)).reset_index(drop=True)

  if rank_print:
      print(f"Select {len(test_dataset)} test data for pseudo-training")

  for violation_type in ["positive", "negative"]:
      for i in range(1, 3):
          sub_dataset = test_dataset[["rule", "subreddit", "positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"]].copy()
          body_col = f"{violation_type}_example_{i}"
          other_positive_col = f"{violation_type}_example_{3-i}"
          sub_dataset["body"] = sub_dataset[body_col]
          sub_dataset[f"{violation_type}_example"] = sub_dataset[other_positive_col]
          anti_violation_type = "negative" if violation_type == "positive" else "positive"
          sub_dataset[f"{anti_violation_type}_example"] = np.where(np.random.rand(len(sub_dataset)) < 0.5, sub_dataset[f"{anti_violation_type}_example_1"],
sub_dataset[f"{anti_violation_type}_example_2"])
          sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
          sub_dataset.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], inplace=True)
          merge.append(sub_dataset)

  return pd.concat(merge, axis=0).drop_duplicates(ignore_index=True)

def build_dataset(df):
  df = df.copy()
  df["prompt"] = df.apply(build_prompt, axis=1)
  columns = ["prompt"]
  if "rule_violation" in df:
      df["completion"] = df["rule_violation"].map({
          1: positive,
          0: negative,})
      columns.append("completion")
  dataset = Dataset.from_pandas(df[columns], preserve_index=False)
  return dataset


Overwriting utils.py


In [34]:
%%writefile train.py
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from trl import SFTTrainer, SFTConfig
from peft import PeftModel, LoraConfig, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils import is_torch_bf16_gpu_available
from transformers import TrainerCallback

from utils import *
from constants import *

class AUCCallback(TrainerCallback):
  def __init__(self, tokenizer):
      self.tokenizer = tokenizer
      self.yes_token_id = tokenizer.encode("Yes", add_special_tokens=False)[-1]
      self.no_token_id = tokenizer.encode("No", add_special_tokens=False)[-1]
      print(f"Yes token ID: {self.yes_token_id}, No token ID: {self.no_token_id}")

  def on_evaluate(self, args, state, control, model, **kwargs):
      print(f"\n{'='*50}")
      print(f"Computing AUC at step {state.global_step}")
      print(f"{'='*50}")

      # Get fresh validation data
      eval_df = get_eval_df()
      eval_dataset = build_dataset(eval_df)

      model.eval()
      all_probs = []
      all_labels = []

      # Process in small batches
      batch_size = 2
      for i in range(0, len(eval_dataset), batch_size):
          batch = eval_dataset[i:min(i+batch_size, len(eval_dataset))]

          prompts = batch["prompt"] if isinstance(batch["prompt"], list) else [batch["prompt"]]
          completions = batch.get("completion", [])

          if len(completions) == 0:
              print(f"No completions found in batch {i}")
              continue

          completions = completions if isinstance(completions, list) else [completions]

          # Get true labels
          true_labels = [1 if str(c).strip() == "Yes" else 0 for c in completions]

          # Get model predictions
          with torch.no_grad():
              inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048)
              inputs = {k: v.to(model.device) for k, v in inputs.items()}

              outputs = model(**inputs)
              logits = outputs.logits

              # Get last non-padding token logits
              for j in range(len(prompts)):
                  seq_len = inputs['attention_mask'][j].sum().item()
                  last_logits = logits[j, seq_len-1, :]

                  yes_no_logits = last_logits[[self.yes_token_id, self.no_token_id]]
                  probs = torch.softmax(yes_no_logits, dim=-1)
                  yes_prob = probs[0].cpu().item()

                  all_probs.append(yes_prob)
                  all_labels.append(true_labels[j])

      try:
          all_labels_array = np.array(all_labels)
          all_probs_array = np.array(all_probs)

          print(f"Total samples: {len(all_labels_array)}")
          print(f"Positive samples: {all_labels_array.sum()}")
          print(f"Negative samples: {len(all_labels_array) - all_labels_array.sum()}")
          print(f"Sample probs: {all_probs_array[:5]}")
          print(f"Sample labels: {all_labels_array[:5]}")

          if len(np.unique(all_labels_array)) < 2:
              print("Warning: Only one class present in labels")
          else:
              auc = roc_auc_score(all_labels_array, all_probs_array)
              print(f"Validation AUC: {auc:.4f}")
      except Exception as e:
          print(f"Error computing AUC: {e}")
          import traceback
          traceback.print_exc()

      print(f"{'='*50}\n")
      model.train()

def main():
  from datasets import concatenate_datasets
  import torch.distributed as dist
  train_dataset = build_dataset(get_df())
  eval_dataset = build_dataset(get_eval_df())

 # Tell trainer to shard data across GPUs properly
  if dist.is_initialized():
      world_size = dist.get_world_size()
      rank = dist.get_rank()

      # Manual sharding: each GPU gets every Nth sample
      train_indices = list(range(rank, len(train_dataset), world_size))
      train_dataset = train_dataset.select(train_indices)

      if rank == 0:
          print(f"Sharding: {len(train_dataset)*world_size} total -> {len(train_dataset)} per GPU")

  lora_config = LoraConfig(
      r=64,
      lora_alpha=128,
      lora_dropout=0.1,
      bias="none",
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
      task_type="CAUSAL_LM",
  )

  training_args = SFTConfig(
      num_train_epochs=1,
      per_device_train_batch_size=4,
      gradient_accumulation_steps=4,
      optim="paged_adamw_8bit",
      learning_rate=1e-4,
      weight_decay=0.01,
      max_grad_norm=1.0,
      lr_scheduler_type="cosine",
      warmup_ratio=0.03,
      fp16=True,
      dataloader_pin_memory=True,
      gradient_checkpointing=True,
      gradient_checkpointing_kwargs={"use_reentrant": False},
      save_strategy="no",
      eval_strategy="steps",
      eval_steps=eval_steps,
      per_device_eval_batch_size=2,
      report_to="none",
      completion_only_loss=True,
      packing=False,
      remove_unused_columns=False,
      prediction_loss_only=True,
  )

 
  model = AutoModelForCausalLM.from_pretrained(
      base_model_path,
      quantization_config=BitsAndBytesConfig(
          load_in_4bit=True,
          bnb_4bit_quant_type="nf4",
          bnb_4bit_compute_dtype=torch.float16,
          bnb_4bit_use_double_quant=True,
      ),
      device_map="balanced_low_0",
      trust_remote_code=True,
      use_cache=False,
  )
  tokenizer = AutoTokenizer.from_pretrained(base_model_path)
  tokenizer.pad_token = tokenizer.eos_token
  
  trainer = SFTTrainer(
      model=model,
      processing_class=tokenizer,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=eval_dataset,
      peft_config=lora_config,
      callbacks=[AUCCallback(tokenizer)],
  )
  trainer.train()
  trainer.save_model(lora_path)

if __name__ == "__main__":
  main()

Overwriting train.py


In [35]:
!accelerate launch --config_file accelerate_config.yaml train.py

[2025-10-23 02:30:27,856] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)
[2025-10-23 02:30:30,104] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
2025-10-23 02:30:30.724139: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761186630.753378    3133 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761186630.765951    3133 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W1023 02:30:37.202000 3133 torch/distributed/run.py:766] 
W1023 02:30:37.202000 3133 torch/distributed/run.py:766] *****************************************
W1023 02:30:37.202000 3133 t

In [36]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  bf16:
    enabled: false
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

Overwriting accelerate_config.yaml


In [37]:
%%writefile inference.py

import os
os.environ["VLLM_USE_V1"] = "0"

import random
import vllm
import torch
import numpy as np
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import *
import multiprocessing as mp

def run_inference_on_device(df_slice):
    llm = vllm.LLM(
        base_model_path,
        quantization="gptq" if use_gptq else None,
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2048,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )
    tokenizer = llm.get_tokenizer()
    outputs = llm.generate(
        build_dataset(df_slice)["prompt"],
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[MultipleChoiceLogitsProcessor(tokenizer, choices=[positive, negative])],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("lora1", 1, lora_path)
    )
    log_probs = [{lp.decoded_token: np.exp(lp.logprob) for lp in out.outputs[0].logprobs[0].values()} for out in outputs]
    predictions = pd.DataFrame(log_probs)[[positive, negative]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions

def worker(device_id, df_slice, return_dict):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")
    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds

def main():
    test_df = pd.read_csv("/kaggle/input/jigsaw-agile-community-rules/test.csv")
    test_df["positive_example"] = test_df.apply(lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]), axis=1)
    test_df["negative_example"] = test_df.apply(lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]), axis=1)
    test_df = test_df.drop(columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"], errors="ignore")

    mid = len(test_df) // 2
    df0 = test_df.iloc[:mid].reset_index(drop=True)
    df1 = test_df.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)
    submission = predictions[["row_id", positive]].rename(columns={positive: "rule_violation"})
    submission.to_csv("/kaggle/working/submission.csv", index=False)

if __name__ == "__main__":
    main()

Overwriting inference.py


In [ ]:
!python inference.py

import pandas as pd
pd.read_csv('/kaggle/working/submission.csv')

2025-10-23 02:32:51.239163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761186771.263709    3336 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761186771.270809    3336 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[Worker 0] Running on GPU 0, data size=5
[Worker 1] Running on GPU 1, data size=5
INFO 10-23 02:32:55 [__init__.py:235] Automatically detected platform cuda.
INFO 10-23 02:32:55 [__init__.py:235] Automatically detected platform cuda.
`torch_dtype` is deprecated! Use `dtype` instead!
WARNING 10-23 02:33:11 [config.py:3443] Casting torch.bfloat16 to torch.float16.
INFO 10-23 02:33:11 [config.py:1604] Using max model len 2048
`torch_dty